# RealThor Evaluation Suite

Your goal: evaluate three prompt versions of RealThor and determine which one, if any, is safe to ship.

RealThor now returns **structured output** (a Pydantic object) instead of free-form text. Evaluation is exact field comparison — no string parsing.

**What you need to do in this notebook:**
1. Implement `ask_realthor()` — calls the structured LLM with a given prompt
2. Implement `evaluate_output()` — compares `priority` and `recommended_action` against the golden dataset
3. Run the evaluation loop over all 20 scenarios × all three prompt versions
4. Build the comparison summary table
5. Implement the decision gate: accept or reject each candidate version

## Setup

In [ ]:
import os
import pandas as pd
from typing import Literal
from pydantic import BaseModel
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
load_dotenv()

In [ ]:
with open("../prompts/system_prompt_v1.md") as f:
    PROMPT_V1 = f.read()

with open("../prompts/system_prompt_v2.md") as f:
    PROMPT_V2 = f.read()

with open("../prompts/system_prompt_v3.md") as f:  # create this file first (Task 1)
    PROMPT_V3 = f.read()

demand = pd.read_csv("../data/demand_signals.csv")
supply = pd.read_csv("../data/supply_snapshot.csv")
scenarios = pd.read_csv("eval_golden_dataset.csv")
display(scenarios)

## Output Schema and Structured LLM

RealThor v0.2 returns a `ListingRecommendation` object. The schema is pre-defined — your evaluation functions will compare `result.priority` and `result.recommended_action` against the golden dataset.

In [ ]:
class ListingRecommendation(BaseModel):
    """Structured recommendation from RealThor."""
    priority: Literal["high", "medium", "low"]
    demand_reasoning: str
    supply_reasoning: str
    recommended_action: Literal["prioritize_new_listings", "monitor_opportunity", "deprioritize"]
    confidence: Literal["high", "medium", "low"]

In [ ]:
provider = os.getenv("LLM_PROVIDER", "openai")
model_kwargs = {"model": "gpt-4o-mini", "temperature": 0}
if provider == "vocareum":
    model_kwargs |= {"base_url": "https://openai.vocareum.com/v1", "api_key": os.getenv("VOCAREUM_API_KEY")}
llm = ChatOpenAI(**model_kwargs)
structured_llm = llm.with_structured_output(ListingRecommendation)

## Helper Functions

`build_context()` is provided. Implement `ask_realthor()` and `evaluate_output()`.

In [ ]:
def build_context(region: str, property_type: str, bedrooms: int) -> str:
    d = demand[
        (demand["region"] == region)
        & (demand["property_type"] == property_type)
        & (demand["bedrooms"] == bedrooms)
    ]
    s = supply[
        (supply["region"] == region)
        & (supply["property_type"] == property_type)
        & (supply["bedrooms"] == bedrooms)
    ]
    if d.empty or s.empty:
        return "No data available for the requested segment."
    return (
        f"Region: {region}\n"
        f"Property: {bedrooms}-bedroom {property_type}\n\n"
        f"Demand signals:\n{d.to_string(index=False)}\n\n"
        f"Supply snapshot:\n{s.to_string(index=False)}"
    )

In [ ]:
def ask_realthor(region: str, property_type: str, bedrooms: int, prompt: str) -> ListingRecommendation:
    """Call structured_llm with the given prompt and return a ListingRecommendation.

    Use build_context() to build the user message.
    Pass the system prompt via SystemMessage and the context via HumanMessage.
    """
    # TODO: implement this function
    pass

In [ ]:
def evaluate_output(result: ListingRecommendation, row) -> dict:
    """Compare result fields against the golden dataset row.

    Check 1 — priority: result.priority == row['expected_priority']
    Check 2 — action:   result.recommended_action == row['expected_action']

    Return: {'passed': bool, 'priority': str, 'action': str, 'failure_reason': str}
    """
    # TODO: implement this function
    pass

## Run Evaluation

Run every scenario against all three prompt versions. Build a `results_df` with columns:
`scenario_id`, `prompt_version`, `passed`, `priority`, `action`, `failure_reason`.

In [ ]:
all_results = []

# TODO: loop over [("v1", PROMPT_V1), ("v2", PROMPT_V2), ("v3", PROMPT_V3)] and all scenarios
# For each combination, call ask_realthor() and evaluate_output()
# Append a dict with scenario_id, prompt_version, and the evaluate_output() fields

results_df = pd.DataFrame(all_results)
display(results_df)

## Comparison Summary

In [ ]:
# TODO: group results_df by prompt_version
# Compute: total scenarios, passed count, pass_rate
# Store in `summary` and display it

## Decision Gate

A candidate version ships only if it meets both conditions:
1. **No regressions** — no scenario that passed in v1 fails in the candidate
2. **Meaningful improvement** — pass rate is at least 10 percentage points above v1

Check both `v2` and `v3`.

In [ ]:
# TODO: implement the decision gate
#
# 1. Build v1_passing: set of scenario_ids that passed in v1
# 2. For each candidate version (v2, v3):
#    a. Find regressions: scenario_ids that passed in v1 but fail in the candidate
#    b. accept = candidate_pass_rate >= v1_pass_rate + 0.10 and len(regressions) == 0
#       (candidate must beat v1 by at least 10 percentage points AND introduce no regressions)
# 3. Print pass rates, regressions, and the decision for each candidate:
#    "ACCEPT" or "REJECT"
# 4. Print the final shipping decision:
#    "Ship v3." / "Ship v2." / "Do not ship. Keep v1."